[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EstebanGZam/taller_agente/blob/main/Taller1_SECOP_Colab.ipynb)

# Taller 1 · Agente analista del SECOP

Este cuaderno reproduce la entrega completa del Taller 1:

1. Clonar el repositorio e instalar dependencias.
2. Cargar las claves desde los secretos de Colab.
3. Pruebas sin modelo de lenguaje (sin costo).
4. Una pregunta de ejemplo al agente.
5. Experimentos A (línea base) vs. B (mejorada) en LangSmith y tabla comparativa.
6. Análisis de aptitud de los 20 procesos para `empresa_multiproposito.json`.
7. Frontend en Gradio.
8. Dónde revisar trazas, costos y latencia en LangSmith.

Criterio del taller: **al menos 85 % de precisión con el menor costo posible**.

## 1. Clonar el repositorio e instalar dependencias

In [ ]:
import os

REPO = "https://github.com/EstebanGZam/taller_agente"
if not os.path.exists("/content/taller_agente"):
    !git clone -q -b main {REPO} /content/taller_agente
%cd /content/taller_agente
!git pull -q
!pip install -q -r requirements.txt pytest pypdf "gradio>=5,<6"

## 2. Claves y configuración

Guarden `ANTHROPIC_API_KEY` y `LANGSMITH_API_KEY` en **Secretos** de Colab (icono de llave en la barra
izquierda) y den acceso a este cuaderno. Si no están, se piden con `getpass`. Las claves nunca se imprimen.

No usen `LANGSMITH_AGENT_ID` ni `LANGSMITH_AGENT_ENVIRONMENT`: junto con `LANGSMITH_PROJECT` impiden
registrar las trazas, por eso se eliminan aquí.

In [ ]:
import os
from getpass import getpass

for variable in ("LANGSMITH_AGENT_ID", "LANGSMITH_AGENT_ENVIRONMENT"):
    os.environ.pop(variable, None)


def secreto(nombre: str) -> str:
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
        if valor:
            return valor
    except Exception:  # fuera de Colab, o el secreto no existe / no tiene acceso
        pass
    return getpass(f"{nombre}: ")


for clave in ("ANTHROPIC_API_KEY", "LANGSMITH_API_KEY"):
    if not os.environ.get(clave):
        os.environ[clave] = secreto(clave)

EQUIPO = "mi-equipo"  # ← cambien por el nombre del equipo
os.environ.update({
    "EQUIPO": EQUIPO,
    "LANGSMITH_TRACING": "true",
    "LANGSMITH_PROJECT": f"secop-taller1-{EQUIPO}",
    "MODELO": "anthropic:claude-haiku-5-5",
    "MODO_DATOS": "muestra",
    "CONFIG": "B",
})
print("Claves cargadas:", {k: bool(os.environ.get(k)) for k in ("ANTHROPIC_API_KEY", "LANGSMITH_API_KEY")})
print("Proyecto de LangSmith:", os.environ["LANGSMITH_PROJECT"])

## 3. Pruebas sin modelo de lenguaje

`herramientas.py` prueba las herramientas en Python puro y `pytest` verifica, sin claves ni red, que las
herramientas devuelven los datos de cada pregunta de referencia, que la configuración A es idéntica al
agente base y los 20 veredictos de aptitud.

In [ ]:
!python herramientas.py
!python -m pytest -q

## 4. Una pregunta al agente

`preguntar()` devuelve la respuesta y las herramientas que el agente decidió usar. `ContadorCostos`
(de `costos.py`) suma los tokens y el costo con la tarifa pública de Anthropic.

In [ ]:
import agente
from costos import ContadorCostos

contador = ContadorCostos(agente.MODELO)
r = agente.preguntar("¿Nuestro capital de trabajo alcanza el 40 % del presupuesto del proceso "
                     "FLO-INFRA-SA-MC-007-2025?", callbacks=[contador])
print(r["respuesta"])
print("\nHerramientas usadas:", r["herramientas"] or "ninguna")
print(f"Configuración {agente.CONFIG} · {agente.MODELO} ·", contador.resumen())

## 5. Experimentos: configuración A vs. B

- **A**: prompt y descripciones originales del curso (línea base).
- **B**: descripciones precisas, salida compacta con resumen y alertas de datos, prompt con reglas por tipo
  de pregunta.

Cada corrida evalúa las 18 preguntas de referencia en LangSmith (dataset `secop-taller1-referencia`) con
dos evaluadores (uso de herramientas y respuesta correcta según un juez) y guarda
`resultados/experimento_<prefijo>.json` con precisión, tokens, costo y latencia.

In [ ]:
!CONFIG=A python evaluar.py "A-base"
!CONFIG=B python evaluar.py "B-mejorado"

In [ ]:
from IPython.display import Markdown, display

!python resumen_experimentos.py A-base B-mejorado > /dev/null
display(Markdown(open("resultados/comparacion_configuraciones.md", encoding="utf-8").read()))

## 6. Aptitud de la empresa en los 20 procesos

Lee los 20 PDF, extrae los requisitos, los verifica contra `empresa_multiproposito.json` y compara el
veredicto (APTA, NO APTA, NO APLICA) con la clave `datos/clave_aptitud.json`. Registra nombre, referencia
y costo de cada proceso en `resultados/aptitud_procesos.{csv,md,json}`.

In [ ]:
from IPython.display import Markdown, display

!python analizar_procesos.py --empresa empresa_multiproposito.json
display(Markdown(open("resultados/aptitud_procesos.md", encoding="utf-8").read()))

In [ ]:
!python evaluar_aptitud.py "apt-haiku"

## 7. Frontend

Abre la interfaz de Gradio con un enlace público temporal (`share=True`). Detengan la celda para cerrarla.

In [ ]:
import app

app.demo.queue().launch(share=True, debug=True)

## 8. Dónde mirar en LangSmith

En [smith.langchain.com](https://smith.langchain.com):

- **Projects → `secop-taller1-<equipo>`**: una traza por pregunta del agente (sección 4 y frontend), con el
  árbol razonar–actuar, las herramientas llamadas, tokens y latencia de cada paso.
- **Datasets & Experiments → `secop-taller1-referencia`**: los experimentos `A-base-…` y `B-mejorado-…`.
  Cada fila muestra la salida (con `metricas`: tokens, costo y latencia) y las columnas
  `uso_herramientas`, `respuesta_correcta` y `costo_usd`.
- Seleccionen los dos experimentos y pulsen **Compare** para ver lado a lado las respuestas, los
  puntajes y las columnas de **Latency**, **Tokens** y **Cost**. Si LangSmith no conoce la tarifa del
  modelo, la columna Cost sale vacía; el costo de referencia es el de `costos.py` (columna `costo_usd` y
  `resultados/comparacion_configuraciones.md`).
- En los metadatos de cada experimento quedan `config` y `modelo`; en las trazas, las etiquetas
  `config-A`/`config-B`.
- **Datasets & Experiments → `secop-taller1-aptitud`**: el experimento `apt-haiku-…` del análisis de aptitud.